# Demo: Running Time-Series Diagnostics

You've fit a model. Great. But how do you know it's any good?

The answer is in the residuals — the errors your model made on the training data. If the model captured all the real patterns, the residuals should look like random noise. If there's structure left in the residuals, the model missed something.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

## Setup — fit both models

We need the fitted models from the previous demo. Let's just re-fit them quickly and move on to the diagnostics.

In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date").asfreq("MS")
train_df = df.loc[:"2023-12-01"]
train_demand = train_df["demand_mwh"]
train_temp = train_df["avg_temp_f"]

arima_fit = SARIMAX(train_demand, order=(2, 1, 1)).fit(disp=False)
sarimax_fit = SARIMAX(
    train_demand, exog=train_temp,
    order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
).fit(disp=False)

print("Both models fitted. Let's look at the residuals.")

## Step 1: Plot the residuals

What should residuals look like? Random noise bouncing around zero. No trends, no seasonal humps, no growing variance.

If you see a pattern, the model missed it.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(arima_fit.resid, color="tab:blue", linewidth=0.8)
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("ARIMA(2,1,1) residuals")
axes[0].set_ylabel("Residual (MWh)")

axes[1].plot(sarimax_fit.resid, color="tab:orange", linewidth=0.8)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_title("SARIMAX(1,1,1)(1,1,1,12) + temp residuals")
axes[1].set_ylabel("Residual (MWh)")

plt.tight_layout()
plt.show()

## Step 2: ACF of residuals

The time-series plot gives you a rough sense, but the ACF is more precise. It shows whether residuals at different lags are correlated.

For ARIMA, look for spikes at lag 12, 24 — those are seasonal patterns the model didn't capture. For SARIMAX, those spikes should be gone (or at least inside the confidence bands).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(arima_fit.resid, ax=axes[0], lags=24)
axes[0].set_title("ARIMA residual ACF")

plot_acf(sarimax_fit.resid, ax=axes[1], lags=24)
axes[1].set_title("SARIMAX residual ACF")

plt.tight_layout()
plt.show()

## Step 3: Ljung-Box test

The ACF plot is visual. The Ljung-Box test is the formal version — it tests whether the residuals have any significant autocorrelation up to a given lag.

We test at lag 12 (one full seasonal cycle). The rule:
- **p > 0.05**: residuals look like white noise. Good.
- **p < 0.05**: there's still structure in the residuals. The model missed something.

In [ ]:
lb_arima = acorr_ljungbox(arima_fit.resid, lags=[12], return_df=True)
lb_sarimax = acorr_ljungbox(sarimax_fit.resid, lags=[12], return_df=True)

print("Ljung-Box test at lag 12")
print(f"  ARIMA     p-value: {lb_arima['lb_pvalue'].values[0]:.4f}")
print(f"  SARIMAX   p-value: {lb_sarimax['lb_pvalue'].values[0]:.4f}")

## Step 4: Normality test

Prediction intervals assume the residuals are roughly normal (bell-shaped). If they're not, the intervals might be too narrow or too wide — a "95%" interval might only cover 85% of actual outcomes.

We use the D'Agostino-Pearson test. Same deal: p > 0.05 means normal enough.

In [ ]:
_, norm_p_arima = stats.normaltest(arima_fit.resid.dropna())
_, norm_p_sarimax = stats.normaltest(sarimax_fit.resid.dropna())

print("Normality test")
print(f"  ARIMA     p-value: {norm_p_arima:.4f}")
print(f"  SARIMAX   p-value: {norm_p_sarimax:.4f}")

## Step 5: Compare AIC and BIC

AIC and BIC both measure how well the model fits the data, with a penalty for complexity. Lower is better.

AIC is a bit more lenient — it'll pick a slightly bigger model. BIC penalizes extra parameters more harshly. If both agree that SARIMAX is better, that's a strong signal.

In [ ]:
print(f"{'Model':<35} {'AIC':>10} {'BIC':>10}")
print("-" * 57)
print(f"{'ARIMA(2,1,1)':<35} {arima_fit.aic:>10.1f} {arima_fit.bic:>10.1f}")
print(f"{'SARIMAX(1,1,1)(1,1,1,12) + temp':<35} {sarimax_fit.aic:>10.1f} {sarimax_fit.bic:>10.1f}")
print()
print(f"SARIMAX wins by {arima_fit.aic - sarimax_fit.aic:.1f} AIC points.")
print("Temperature and seasonality actually help — the extra complexity is worth it.")

## Takeaway

Diagnostics aren't optional. A model can give you a forecast and still be garbage. The residual checks tell you whether to trust the output.

Quick checklist:
1. Residuals look like noise? Check the plot and ACF.
2. Ljung-Box p > 0.05? No leftover autocorrelation.
3. Normality p > 0.05? Prediction intervals are trustworthy.
4. AIC/BIC lower than the alternative? The model is worth its complexity.

In the exercise, you'll run these same checks on your own fitted models.